# Brique 0 — Comprendre Python avant d'écrire une ligne de code

On va construire ensemble, depuis zéro, ce qui se passe vraiment quand tu écris et lances un programme Python.
Je ne vais rien supposer connu : chaque mot un peu technique sera défini en langage simple avant d'être utilisé.
Si à un moment un mot te semble flou, c'est le signe que je n'ai pas assez détaillé — c'est à corriger, pas à toi
de deviner.

On va avancer dans cet ordre, chaque étape s'appuyant sur la précédente :

1. Pourquoi ce notebook existe, et pourquoi ça compte vraiment
2. Ce qu'un ordinateur comprend réellement (avant même de parler de Python)
3. Les deux grandes stratégies pour exécuter un programme : compiler, ou interpréter
4. Où se situe Python dans tout ça — ni tout à fait l'un, ni tout à fait l'autre
5. Un petit détour par l'histoire : qui a créé Python, pourquoi, et avec quelle philosophie
6. Ce que "CPython" veut dire, et pourquoi il existe d'autres versions de Python
7. Le typage dynamique : la différence la plus concrète avec Java, construite pas à pas
8. Le "duck typing", une conséquence directe du typage dynamique
9. Le GIL (Global Interpreter Lock) — un sujet souvent mal expliqué, qu'on va démonter calmement
10. Comment Python gère la mémoire, sans que tu aies à t'en occuper toi-même
11. Pour aller plus loin si tu veux creuser

## 1. Pourquoi ce notebook existe

Tu sais déjà programmer : tu as fait pas mal de Java pendant ta scolarité. Le risque, quand on connaît déjà un
langage et qu'on en apprend un deuxième, c'est de rester sur ses réflexes et de ne jamais comprendre *pourquoi*
le nouveau langage fait certains choix différemment. On code alors "au feeling", et le jour où quelque chose se
comporte de façon inattendue (une erreur bizarre, une lenteur, un comportement different entre deux bouts de
code qui semblaient identiques), on est bloqué, parce qu'on n'a pas de modèle mental de ce qui se passe
réellement derrière.

Ce notebook construit ce modèle mental. Pas pour que tu deviennes expert en fonctionnement interne tout de
suite, mais pour que la suite du parcours (bonnes pratiques, POO, algorithmique, ML) s'appuie sur une base
solide plutôt que sur des recettes apprises par cœur sans les comprendre.

## 2. Ce qu'un ordinateur comprend réellement

Avant de parler de Python ou même de Java, il faut repartir d'un fait de base, qu'on oublie vite une fois qu'on
sait programmer : **le processeur de ton ordinateur (le composant qui exécute réellement les instructions) ne
comprend qu'un seul "langage"**, et ce langage n'a rien à voir avec `print("bonjour")` ou `System.out.println`.

Ce langage s'appelle le **code machine**. C'est une suite de nombres (en binaire, donc des 0 et des 1), où
chaque petite suite de nombres correspond à une instruction extrêmement basique et extrêmement précise : "prends
la valeur qui est à cet endroit précis de la mémoire, additionne-la à celle qui est à cet autre endroit précis,
range le résultat ici". Rien de plus abstrait que ça. Pas de "variable qui s'appelle `age`", pas de "fonction",
juste des emplacements mémoire numérotés et des opérations très élémentaires sur ces emplacements.

Aucun humain ne programme directement comme ça au quotidien (c'est possible, ça s'appelle écrire en
**assembleur**, mais c'est extrêmement lent et pénible pour écrire un vrai programme). On a donc inventé des
**langages de programmation** — Python, Java, C, etc. — qui permettent d'écrire des instructions beaucoup plus
proches de la façon dont un humain raisonne (`si l'âge est supérieur à 18, alors...`), avec des noms de variables
lisibles, des fonctions, des boucles.

Le problème, c'est que le processeur ne comprend toujours que le code machine, pas `age > 18`. **Il faut donc
forcément, à un moment, transformer ce que tu as écrit en code machine.** C'est précisément le rôle de la section
suivante : il existe deux grandes façons de faire cette transformation, et Python comme Java ont chacun fait un
choix (en fait, un choix hybride, on va le voir).

## 3. Les deux grandes stratégies : compiler, ou interpréter

Imagine que tu as écrit un livre en français, et que tu veux le faire lire à quelqu'un qui ne parle qu'anglais.
Il y a deux façons très différentes de procéder.

**Stratégie 1 : la traduction complète à l'avance (la compilation).**
Tu donnes le livre entier à un traducteur professionnel. Il prend le temps de tout traduire, une bonne fois pour
toutes, et te rend un livre entièrement en anglais. Ensuite, n'importe qui qui parle anglais peut le lire
directement, rapidement, sans traducteur à côté de lui. Mais si tu changes une seule phrase dans le livre
original, il faut refaire traduire (au moins cette partie).

En programmation, cette étape de traduction complète s'appelle la **compilation**. Le programme qui fait cette
traduction s'appelle un **compilateur**. Le résultat de cette traduction (le "livre en anglais") est un fichier
en code machine, directement exécutable par le processeur, qu'on appelle un **exécutable**. C'est la stratégie
du langage C, par exemple : tu écris ton code, tu le compiles une fois, tu obtiens un fichier exécutable que tu
peux lancer autant de fois que tu veux sans retraduire à chaque fois.

**Stratégie 2 : la traduction en direct, phrase par phrase (l'interprétation).**
Au lieu de traduire le livre entier à l'avance, tu amènes un traducteur **en personne**, à côté du lecteur
anglophone, et à chaque phrase que le lecteur veut connaître, le traducteur la traduit à voix haute, sur le
moment. Rien n'est traduit à l'avance. C'est plus lent à l'usage (il faut un traducteur présent à chaque lecture),
mais c'est plus souple : si tu changes une phrase du livre, pas besoin de retraduire quoi que ce soit avant, le
traducteur s'adaptera simplement au moment venu.

En programmation, ce traducteur en direct s'appelle un **interpréteur**. Il lit ton code source (ton texte
`.py` ou `.java`, écrit par un humain) et l'exécute directement, instruction par instruction, sans produire de
fichier exécutable séparé au préalable.

**Pourquoi cette différence compte concrètement** : un programme compilé démarre en général plus vite à
l'exécution (la traduction est déjà faite), mais il faut une étape de compilation avant de pouvoir le lancer. Un
programme interprété peut être lancé immédiatement depuis son code source, mais chaque exécution refait un
travail de traduction, ce qui peut le rendre plus lent.

## 4. Où se situe Python ? Ni tout à fait l'un, ni tout à fait l'autre

Si on te demande "Python est-il compilé ou interprété ?", la vraie réponse honnête est : **un peu des deux, dans
cet ordre**. Construisons ça étape par étape.

**Étape 1 — une traduction partielle d'abord.** Quand tu exécutes un fichier `.py`, Python ne lit pas ton texte
ligne par ligne directement comme le ferait un traducteur humain lisant du français brut. Il fait d'abord une
petite traduction préalable, vers un langage intermédiaire beaucoup plus simple et beaucoup plus proche du code
machine que ton Python original, mais qui n'est **toujours pas** du vrai code machine. Ce langage intermédiaire
s'appelle le **bytecode**.

Pour reprendre l'analogie du livre : c'est comme si, avant de donner le livre au traducteur en direct, on le
faisait d'abord passer par une réécriture dans une version simplifiée et standardisée du français — pas encore
de l'anglais, mais une forme beaucoup plus facile et rapide à traduire à la volée ensuite, parce que toutes les
tournures compliquées ont déjà été aplaties.

**Étape 2 — ce bytecode est mis de côté.** Cette traduction intermédiaire est enregistrée dans un fichier séparé
(tu verras parfois apparaître un dossier `__pycache__/` contenant des fichiers `.pyc` — ce sont exactement ces
bytecodes enregistrés). Le but : si tu relances le programme sans avoir modifié le fichier source, Python n'a
pas besoin de refaire cette première traduction, il réutilise celle déjà faite. C'est un gain de temps, pas une
étape obligatoire conceptuellement.

**Étape 3 — un programme exécute ce bytecode, instruction par instruction.** C'est là qu'intervient le vrai
"traducteur en direct" : un programme (écrit, lui, en langage C) appelé la **machine virtuelle Python**, qui lit
ce bytecode et exécute, une par une, les actions qu'il décrit. C'est cette machine virtuelle qui fait vraiment
le travail d'interprétation en direct — mais sur le bytecode déjà simplifié, pas sur ton texte Python original.

**Donc, pour résumer en une phrase** : Python compile d'abord ton code vers un langage intermédiaire simplifié
(le bytecode), puis interprète ce bytecode en direct. Ce n'est ni une pure compilation (il n'y a jamais de
fichier final en vrai code machine produit), ni une interprétation naïve du texte brut.

Vérifions cette idée concrètement. La cellule ci-dessous utilise un outil intégré à Python, le module `dis`
(abréviation de "disassemble", "désassembler"), qui permet de regarder ce bytecode intermédiaire pour une
fonction donnée.

In [ ]:
import dis

def addition(a, b):
    return a + b

dis.dis(addition)

Ce que tu vois affiché n'est **ni** ton code Python original (`return a + b`), **ni** du vrai code machine
illisible (des nombres binaires) : c'est exactement l'étape intermédiaire dont on vient de parler, le bytecode,
affiché sous une forme lisible par un humain pour qu'on puisse l'inspecter.

Tu n'as pas besoin de comprendre le sens exact de chaque ligne (`LOAD_FAST`, `BINARY_OP`, `RETURN_VALUE`...).
Retiens juste l'idée : **chaque instruction Python que tu écris est décomposée en plusieurs petites étapes bien
plus élémentaires**, dans cet ordre : aller chercher la valeur de `a`, aller chercher la valeur de `b`, faire
l'addition, renvoyer le résultat. C'est exactement ce niveau de détail dont le processeur a besoin, et c'est ce
détour qui permet d'y arriver sans que toi, en tant que développeur, tu aies à l'écrire toi-même.

**Petite expérience à faire toi-même** : modifie la cellule au-dessus, remplace `a + b` par `a * b`, puis par
`a if a > b else b`, et relance-la chaque fois. Observe comment le nombre et le type d'instructions changent.
Le but n'est pas de tout comprendre, juste de voir de tes propres yeux que du texte Python différent produit du
bytecode différent — la preuve que cette étape de traduction est bien réelle et pas juste une explication
théorique.

**Et Java, dans tout ça ?** Tu vas voir que c'est en fait très proche, ce qui va te rassurer : tu connais déjà
une bonne partie de ce principe sans le savoir.

Quand tu compiles un fichier `MonProgramme.java`, tu obtiens un fichier `MonProgramme.class`. Ce fichier `.class`
n'est **pas** du code machine directement exécutable par ton processeur : c'est, lui aussi, un bytecode
intermédiaire (le bytecode de la JVM, la "machine virtuelle Java"). Et c'est la JVM, au moment où tu lances ton
programme avec `java MonProgramme`, qui lit ce bytecode et l'exécute — exactement le même principe en trois
étapes que pour Python (code source → bytecode → exécution par une machine virtuelle).

**Alors où est la vraie différence entre Java et Python ?** Elle est là : depuis longtemps, la JVM contient en
plus un composant appelé un **JIT** ("Just-In-Time compiler", compilateur "juste à temps"). Ce composant
surveille quelles parties du bytecode sont exécutées très souvent (par exemple une boucle lancée des millions de
fois), et pour **ces parties-là seulement**, il fait l'effort supplémentaire de les traduire une bonne fois pour
toutes en vrai code machine natif, pour que les exécutions suivantes soient beaucoup plus rapides — un peu comme
un traducteur en direct qui, après avoir traduit cent fois la même phrase, décide de la mémoriser une fois pour
toutes plutôt que de la retraduire à chaque fois.

CPython (l'implémentation de Python que tu utilises — on détaille ce mot dans la section suivante) n'a
historiquement **pas** ce mécanisme de JIT : il réexécute le bytecode à chaque fois de la même façon, sans
jamais le convertir en code machine natif. C'est l'une des raisons pour lesquelles du code Python pur est souvent
plus lent que du code Java équivalent sur des calculs répétés intensivement — et c'est aussi pour ça que des
bibliothèques comme NumPy (que tu verras en Brique 4) existent : elles délèguent le calcul lourd à du code C déjà
compilé en code machine, pour contourner complètement cette limite.

## 5. Un peu d'histoire : qui, pourquoi, et avec quel état d'esprit

Maintenant qu'on sait ce que fait Python techniquement, prenons deux minutes pour savoir d'où il vient — ça
éclaire pourquoi le langage est conçu comme il est.

Python est créé par **Guido van Rossum**, un chercheur néerlandais qui travaillait à l'époque dans un institut de
recherche à Amsterdam (le CWI). Il commence à l'écrire en décembre 1989, et la première version est publiée en
1991. Le nom "Python" ne vient pas du serpent : il vient de l'émission comique britannique *Monty Python's
Flying Circus*, que Guido van Rossum aimait beaucoup. Il voulait, selon ses propres mots, un nom "court, unique,
et légèrement mystérieux".

Avant Python, Guido van Rossum avait travaillé sur un autre langage, appelé **ABC**, conçu pour être facile à
enseigner et à apprendre (d'où l'idée de l'indentation qui structure le code, qu'on retrouve ensuite en Python).
Le problème d'ABC, c'est qu'il était difficile d'étendre ses capacités (impossible d'ajouter facilement de
nouvelles fonctionnalités ou bibliothèques). Python reprend les bonnes idées d'ABC (la lisibilité, la
simplicité), tout en corrigeant ce défaut : on peut écrire ses propres modules, importer des bibliothèques
écrites en C, etc.

**Les grandes dates à connaître**, parce qu'elles reviennent souvent en discussion :

| Version | Année | Ce que ça a changé |
|---|---|---|
| Python 1.0 | 1994 | Premier langage "complet", utilisable sérieusement |
| Python 2.0 | 2000 | Ajout d'un vrai ramasse-miettes moderne (voir section 10), et des "list comprehensions" (vues en Brique 1) |
| Python 3.0 | 2008 | Une rupture **volontaire** de compatibilité avec Python 2, pour corriger des défauts de conception qu'on ne pouvait pas changer sans casser l'ancien code (par exemple, en Python 2, `print` n'était pas une fonction normale, et la division entre deux nombres entiers donnait parfois des résultats surprenants) |
| 1er janvier 2020 | — | Fin officielle du support de Python 2. Si tu croises un jour du code avec `print "quelque chose"` sans parenthèses, c'est un signe très clair que c'est du vieux code Python 2 |

**La philosophie du langage** est résumée dans un petit texte caché, accessible directement depuis Python
lui-même (un "easter egg" officiel, une blague intégrée volontairement par les créateurs du langage). Essaie :

In [ ]:
import this

Ce texte s'appelle le "Zen de Python", 19 principes de conception. Trois d'entre eux vont revenir souvent dans ce
parcours, donc autant les comprendre maintenant :

- *"Explicit is better than implicit"* ("explicite vaut mieux qu'implicite") : si un comportement peut être
  écrit clairement plutôt que deviné en silence par le langage, Python préfère qu'il soit écrit clairement.
- *"Simple is better than complex"* ("simple vaut mieux que complexe") : on y reviendra beaucoup en Brique 2
  (POO), notamment pour apprendre à **ne pas** sur-architecturer un programme quand ce n'est pas nécessaire.
- *"Readability counts"* ("la lisibilité compte") : Python est pensé pour être lu par un humain, pas seulement
  pour être exécuté par une machine.

## 6. "CPython" : pourquoi ce mot, et pourquoi il existe d'autres versions de Python

Il faut distinguer deux choses qu'on confond souvent :

- **"Python"** désigne le **langage** : les règles de syntaxe, ce que signifie `for`, `def`, `+`, etc. C'est une
  spécification, un ensemble de règles écrites, pas un programme en soi.
- **"CPython"** désigne un **programme concret**, écrit en langage C, qui sait lire du code respectant les
  règles du langage Python et l'exécuter réellement (en suivant tout le mécanisme décrit en section 4 : code
  source → bytecode → exécution par une machine virtuelle). C'est ce qu'on appelle une **implémentation** du
  langage Python.

Pourquoi cette distinction compte : il est tout à fait possible d'écrire **plusieurs programmes différents**
qui comprennent et exécutent tous le même langage Python, chacun avec ses propres choix d'implémentation. CPython
est de loin le plus utilisé — c'est celui qui démarre quand tu tapes `python` dans un terminal — mais ce n'est
pas le seul :

| Implémentation | Écrite en | Pourquoi elle existe |
|---|---|---|
| **CPython** | C | L'implémentation de référence, celle que (presque) tout le monde utilise par défaut |
| **PyPy** | Python (elle s'exécute elle-même) | Elle a un vrai JIT (voir section 4) → souvent 4 à 10 fois plus rapide que CPython sur du code Python pur |
| **Jython** | Java | Elle produit du bytecode compréhensible par la JVM → permet d'utiliser des bibliothèques Java depuis du code Python |
| **MicroPython** | C | Une version très allégée, conçue pour tourner sur des très petits appareils (microcontrôleurs) |

Tu n'as pas besoin de retenir cette liste par cœur. Retiens juste le principe : si un jour quelqu'un dit "on a
basculé ce service sur PyPy pour la performance", tu sauras maintenant que ça veut dire "même langage Python,
mais exécuté par un programme différent de CPython, qui ajoute un JIT".

In [ ]:
import sys
print(sys.implementation.name)  # te confirme que tu utilises bien CPython
print(sys.version)              # la version précise

## 7. Le typage dynamique, construit pas à pas

C'est probablement la différence la plus concrète que tu vas ressentir en passant de Java à Python, donc on va
prendre le temps de la construire soigneusement.

**Reprenons depuis la définition la plus simple possible d'un "type".** Un type, c'est juste une catégorie qui
dit deux choses : quelle sorte de valeur on a (un nombre entier ? du texte ? une liste ?), et donc ce qu'on a le
droit de faire avec. Un nombre entier, on peut l'additionner. Un texte, on peut le mettre en majuscules. Une
liste, on peut y ajouter un élément. Le type d'une valeur détermine ses capacités.

**En Java, le typage est statique.** Quand tu écris `int age = 20;`, tu dis explicitement, au moment où tu écris
le code, que la variable `age` contiendra toujours un entier. Le compilateur Java vérifie cette promesse **avant
même d'exécuter le programme** : si plus loin dans ton code tu écris `age = "vingt";`, ça ne compile pas, tu as
une erreur immédiate, avant d'avoir rien exécuté. Le type est attaché au **nom de la variable** pour toute la
durée du programme.

**En Python, le typage est dynamique.** Il n'y a pas de déclaration de type à l'avance, et pas de vérification
avant l'exécution. Pour comprendre pourquoi, il faut changer légèrement l'image qu'on a d'une variable.

Imagine une variable Python non pas comme une boîte qui contient une valeur et dont l'étiquette décrit le type
de cette valeur pour toujours (l'image qu'on a souvent en tête venant de Java), mais plutôt comme un **nom
autocollant que tu peux coller sur n'importe quelle boîte**, et que tu peux décoller et recoller sur une boîte
complètement différente à tout moment. Le type n'est jamais attaché au nom (à l'autocollant) : il est attaché à
la boîte elle-même (l'objet réel qui existe en mémoire à cet instant). Le nom, lui, n'a aucun type propre — il se
contente de pointer vers une boîte, et cette boîte peut changer.

Vérifions ça très concrètement :

In [ ]:
x = 5
print(x, "est de type", type(x))   # x pointe actuellement vers un entier

x = "cinq"  # on recolle le même autocollant "x" sur une boîte complètement différente
print(x, "est de type", type(x))   # x pointe maintenant vers une chaîne de caractères

# En Java, l'équivalent de cette deuxième ligne (réaffecter une chaîne à une variable déclarée int)
# ne compile tout simplement pas. En Python, rien n'empêche cette réaffectation :
# aucune vérification de type n'a lieu avant que la ligne ne s'exécute réellement.

**Et l'erreur de type, alors, elle n'existe plus en Python ?** Si, elle existe toujours, mais elle se produit à
un moment différent : non plus avant l'exécution (comme en Java), mais **au moment précis où la ligne
incompatible s'exécute réellement**. Essaie :

In [ ]:
nombre = 5
texte = "cinq"
resultat = nombre + texte  # cette ligne va provoquer une erreur : regarde laquelle, et à quel moment elle apparaît

Tu vois apparaître un `TypeError`, et surtout : **cette erreur n'apparaît qu'au moment où cette ligne précise
s'exécute**, pas avant. Si cette ligne était cachée dans une fonction jamais appelée, Python ne t'aurait jamais
prévenu du problème. C'est exactement la consequence du typage dynamique : rien n'est vérifié à l'avance, tout
est vérifié sur le moment, ligne par ligne, pendant que le programme tourne.

**Pourquoi c'est important de le savoir dès maintenant** : ça veut dire qu'en Python, c'est à toi de faire
attention aux types (ou d'écrire des tests, qu'on verra plus tard dans le parcours), parce que le langage ne te
protège pas automatiquement comme Java le fait. C'est aussi pour ça qu'il existe aujourd'hui des outils optionnels
(les "type hints", qu'on verra en pratique plus tard, qui s'écrivent par exemple `def f(x: int) -> str:`) qui
permettent d'annoter le type attendu, sans que ça change le fonctionnement réel du langage — ce sont de simples
indications, vérifiables par un outil externe (`mypy`), mais que Python lui-même ignore complètement à
l'exécution.

## 8. Le "duck typing" : une conséquence directe du typage dynamique

Maintenant qu'on a bien construit le typage dynamique, on peut comprendre facilement une conséquence directe,
qu'on appelle le **duck typing** ("typage canard"). Le nom vient d'un dicton anglais : *"If it looks like a duck,
swims like a duck, and quacks like a duck, then it probably is a duck"* ("si ça a l'air d'un canard, si ça nage
comme un canard, et si ça cancane comme un canard, alors c'est probablement un canard"). L'idée : on ne se
préoccupe pas de **ce qu'est officiellement** une chose, seulement de **ce qu'elle sait faire**.

**En Java**, pour que deux classes différentes puissent être utilisées de façon interchangeable dans une même
fonction, il faut en général qu'elles partagent explicitement une interface ou une classe mère commune, déclarée
dans le code à l'avance (`class Chat implements Animal`). C'est une vérification faite par le compilateur, basée
sur une parenté déclarée.

**En Python**, à cause du typage dynamique (rien n'est vérifié avant l'exécution), cette contrainte disparaît :
il suffit qu'un objet **possède effectivement** la méthode utilisée, peu importe sa classe, peu importe s'il
existe un quelconque lien de parenté déclaré entre les classes. Voyons ça avec un exemple concret, volontairement
très simple :

In [ ]:
class Canard:
    def parler(self):
        return "Coin coin"

class Humain:
    def parler(self):
        return "Je imite un canard : coin coin"

# Regarde bien : Canard et Humain n'ont AUCUN lien entre elles. Pas d'interface commune,
# pas d'héritage, rien de déclaré à l'avance qui dirait "ces deux classes sont compatibles".

def faire_parler(quelque_chose):
    # cette fonction ne demande jamais "es-tu un Canard, officiellement ?"
    # elle se contente d'essayer d'appeler .parler() sur ce qu'on lui donne, et ça suffit
    print(quelque_chose.parler())

faire_parler(Canard())
faire_parler(Humain())

Les deux appels fonctionnent, sans qu'aucune relation de parenté n'ait jamais été définie entre `Canard` et
`Humain`. La fonction `faire_parler` ne vérifie jamais le type exact de ce qu'on lui donne : elle fait confiance,
et ça échoue seulement si, au moment de l'appel, l'objet donné n'a effectivement pas de méthode `parler()`
(exactement comme pour le `TypeError` de la section précédente : l'erreur, si elle survient, survient à
l'exécution, pas avant).

**C'est une arme à double tranchant.** Ça rend Python très souple à écrire (pas besoin de déclarer des interfaces
pour tout). Mais ça déplace la détection des erreurs de compatibilité de type de la compilation (comme en Java,
où l'erreur est détectée tôt, avant même de lancer le programme) vers l'exécution (potentiellement bien plus
tard, parfois même en production, sur un cas particulier jamais testé). On reparlera beaucoup de cette tension
souplesse/sécurité en Brique 2, quand on construira de vraies architectures orientées objet.

## 9. Le GIL (Global Interpreter Lock) — démonté calmement

C'est le sujet le plus souvent cité au sujet de Python, et aussi le plus souvent mal expliqué. On va construire
chaque mot avant de les assembler.

**D'abord, qu'est-ce qu'un "thread" ("fil d'exécution") ?** Normalement, un programme exécute ses instructions
une par une, dans l'ordre, comme une seule personne qui lit un livre page après page. Un thread, c'est une façon
de faire exécuter **plusieurs séquences d'instructions "en même temps"** au sein du même programme — comme si on
avait plusieurs lecteurs différents qui lisaient simultanément des pages différentes du même livre. Utiliser
plusieurs threads (le "threading", ou "multithreading") peut, en théorie, accélérer un programme sur une machine
qui a plusieurs cœurs de processeur, puisque chaque cœur pourrait exécuter un thread différent réellement en
parallèle, physiquement en même temps.

**Pourquoi un verrou serait-il nécessaire ?** Rappelle-toi la section 10 (mémoire) qu'on va détailler juste après
— en résumé rapide ici : chaque objet en mémoire garde un petit compteur du nombre de "noms" (d'autocollants, cf.
section 7) qui pointent vers lui. Si **deux threads** modifiaient ce compteur **exactement au même instant**,
sans aucune protection, on pourrait obtenir un résultat incohérent : par exemple, les deux threads lisent la
valeur "3", chacun l'incrémente de son côté vers "4" et l'écrit, alors que la bonne valeur finale aurait dû être
"5" (un incrément a été "perdu"). Ce genre de bug, où le résultat dépend d'un ordre d'exécution imprévisible
entre plusieurs threads, s'appelle une **race condition** ("condition de concurrence") — littéralement, les
threads sont en "course" l'un contre l'autre pour accéder à la même donnée, et le résultat dépend de qui arrive
en premier.

**La solution choisie par les créateurs de CPython**, dans les années 1990, et jamais retirée depuis (pour ne
pas casser toute la compatibilité avec le code existant) : un seul verrou global, unique pour tout le programme.
Ce verrou s'appelle le **GIL**. Sa règle est simple et stricte : **à un instant donné, un seul thread a le droit
d'exécuter du bytecode Python**, même si ta machine a 16 cœurs de processeur disponibles. Les autres threads
doivent attendre leur tour.

**Ce que ça veut dire concrètement, selon la situation :**

| Situation | Le GIL empêche-t-il un vrai gain de vitesse ? |
|---|---|
| Plusieurs threads qui font tous du calcul Python pur (boucles, calculs arithmétiques) | Oui — comme un seul thread a le droit d'avancer à la fois, il n'y a aucun gain de vitesse réel, parfois même une perte (le temps passé à faire attendre les threads les uns les autres) |
| Des threads qui attendent surtout une réponse du réseau, du disque, ou d'une base de données (ce qu'on appelle des opérations "d'entrée/sortie", ou "I/O") | Non — pendant qu'un thread attend une réponse extérieure, il relâche volontairement le GIL, ce qui laisse un autre thread travailler pendant ce temps d'attente. Le threading reste donc utile pour ce genre de situation |
| Du code NumPy ou Pandas qui, en dessous, appelle des fonctions écrites en C | Souvent non — ces bibliothèques relâchent elles-mêmes le GIL pendant qu'elles font leur calcul en C, donc un vrai parallélisme redevient possible à l'intérieur de ce calcul |
| Besoin d'un vrai calcul parallèle en Python pur | On utilise alors `multiprocessing` : au lieu de plusieurs threads à l'intérieur d'un même programme, on lance plusieurs **processus** complètement séparés, chacun avec son propre interpréteur Python et donc son propre GIL indépendant — un vrai parallélisme, mais avec un coût (chaque processus a sa mémoire séparée, la communication entre eux est plus lourde) |

**La règle pratique à retenir, simplement** : *threading pour attendre, multiprocessing pour calculer*. On y
reviendra concrètement quand ce sera utile dans les briques suivantes — pas besoin d'en faire plus pour le
moment.

## 10. Comment Python gère la mémoire

**D'abord, le problème général.** Chaque fois que ton programme crée quelque chose (une liste, un nombre, un
objet), cette chose occupe de la place dans la mémoire de l'ordinateur. Cette mémoire est limitée. Si le
programme ne libérait jamais la place occupée par des choses dont il n'a plus besoin, il finirait par saturer
toute la mémoire disponible et planter. Il faut donc un mécanisme pour détecter "cette chose n'est plus utilisée,
on peut récupérer la place qu'elle occupait".

**En Java**, ce mécanisme s'appelle le "garbage collector" (ramasse-miettes) : un processus qui tourne
périodiquement, inspecte l'ensemble de la mémoire utilisée, repère ce qui n'est plus accessible depuis le
programme, et le détruit.

**En Python**, le mécanisme principal est différent, et plus immédiat : on l'appelle le **comptage de
références**. Reprenons l'image de la section 7 (le nom comme un autocollant collé sur une boîte). L'idée du
comptage de références : chaque boîte (chaque objet en mémoire) garde elle-même un petit compteur du nombre
d'autocollants actuellement collés sur elle. Dès que ce compteur retombe à zéro — plus aucun nom ne pointe vers
cette boîte — l'objet est détruit **immédiatement**, sans attendre le passage périodique d'un quelconque
ramasse-miettes.

Vérifions ça concrètement :

In [ ]:
import sys

a = [1, 2, 3]
print(sys.getrefcount(a))  # le compteur de références de l'objet liste pointé par 'a'
# (il affichera au moins 2 : le nom 'a', plus une référence temporaire créée par l'appel
#  à getrefcount lui-même pendant qu'il s'exécute — ignore ce +1, regarde juste comment ça varie)

b = a  # on colle un deuxième autocollant, 'b', sur la MÊME boîte que 'a' (pas une copie : le même objet)
print(sys.getrefcount(a))  # le compteur a augmenté de 1

del b  # on retire l'autocollant 'b'
print(sys.getrefcount(a))  # le compteur est redescendu : on est revenu à la valeur de départ

**Mais le comptage de références seul a une faille.** Imagine deux boîtes, A et B, qui se pointent l'une l'autre
mutuellement (A contient un autocollant qui pointe vers B, et B contient un autocollant qui pointe vers A), et
qu'aucun autre nom, nulle part ailleurs dans le programme, ne pointe plus vers A ou vers B. Ces deux boîtes sont
en réalité devenues inutiles et inaccessibles depuis le reste du programme — mais leur compteur de références
respectif ne tombera **jamais** à zéro, puisqu'elles continuent de se pointer mutuellement l'une l'autre ! On
appelle ça un **cycle de références**. Le simple comptage de références est incapable de détecter ce cas,
puisqu'il ne regarde que "combien de choses pointent vers moi", sans jamais se demander "est-ce que ces
choses-là, elles, sont encore accessibles depuis le reste du programme".

C'est pour corriger précisément ce cas particulier que CPython possède, **en plus** du comptage de références
immédiat, un vrai ramasse-miettes classique (module `gc`), qui tourne seulement périodiquement, et dont le seul
but est de détecter et casser ces cycles. Le comptage de références fait tout le travail courant, en continu et
immédiatement ; le ramasse-miettes périodique ne s'occupe que de ce cas particulier et plus rare.

## 11. Pour aller plus loin, si tu veux creuser

Rien de tout ça n'est obligatoire pour continuer le parcours — c'est juste pour les moments où tu voudrais
creuser un point précis plus en profondeur :

- Documentation officielle, *Design and History FAQ* : https://docs.python.org/3/faq/design.html — l'historique
  du langage écrit directement par ses créateurs.
- Le module `dis`, qu'on a déjà utilisé, a une documentation complète si tu veux comprendre chaque instruction de
  bytecode en détail.
- Si tu veux vraiment descendre dans l'implémentation de CPython ligne par ligne : le livre *CPython Internals*
  d'Anthony Shaw.
- Pour creuser le GIL avec encore plus de détails et d'animations : la conférence (en vidéo) *Understanding the
  Python GIL* de David Beazley, une référence très pédagogique sur ce sujet précis.

---

**Prochaine étape** : `01_pratique.ipynb` — comment structurer un repo Python (les dossiers `src/`, les
fichiers `requirements.txt`, `.env`, `.gitignore`, etc.), maintenant que tu sais *pourquoi* ces outils existent.